<a href="https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of analysis: One row = one content item (content_hash_id) × one client (client_hash_id) × one report_date, from fact_content_daily_performance. My lane uses the GSC-side columns (gsc_impressions, gsc_clicks, gsc_sum_position) to compute ctr_gap. Time window: month = 2026-03.

In [1]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
print(con.sql(f"DESCRIBE SELECT * FROM '{fact_path}' LIMIT 1"))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Fields: feature / label / context / excluded

Feature: gsc_impressions, gsc_clicks, gsc_sum_position (used to derive gsc_avg_position = gsc_sum_position/gsc_impressions and ctr = gsc_clicks/gsc_impressions)
Label (proxy): ctr_gap — derived as expected_ctr(position_tier) − actual_ctr, where expected_ctr is the median ctr of rows sharing the same position tier
Context: client_hash_id, content_hash_id, report_date, gsc_data_available (used for grouping/filtering, not fed to the model)
Excluded: ga4_* columns and ai_* columns (sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events) — my lane is GSC/CTR-based, not engagement/AI-visibility based

In [2]:
q_fields = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(gsc_impressions) AS non_null_impressions,
    COUNT(gsc_clicks) AS non_null_clicks,
    COUNT(gsc_sum_position) AS non_null_position,
    SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS rows_with_gsc_available
FROM '{fact_path}'
"""
print(con.sql(q_fields))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┬─────────────────┬───────────────────┬─────────────────────────┐
│ total_rows │ non_null_impressions │ non_null_clicks │ non_null_position │ rows_with_gsc_available │
│   int64    │        int64         │      int64      │       int64       │         int128          │
├────────────┼──────────────────────┼─────────────────┼───────────────────┼─────────────────────────┤
│    9841378 │              9841378 │         9841378 │           9841378 │                 3611061 │
└────────────┴──────────────────────┴─────────────────┴───────────────────┴─────────────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)
Three checks below: (1) grain probe — must return empty, confirming one row per content×client×day; (2) row count and date span for month=2026-03; (3) availability filter with IS TRUE, showing how many rows have usable GSC data.

In [3]:
print("Grain probe (should be empty):")
print(con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM '""" + fact_path + """'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""))

print("\nRow count + date span:")
print(con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM '{fact_path}'"))

print("\nAvailability (IS TRUE):")
print(con.sql(f"SELECT COUNT(*) AS available_rows FROM '{fact_path}' WHERE gsc_data_available IS TRUE"))


Grain probe (should be empty):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   n   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘


Row count + date span:
┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘


Availability (IS TRUE):
┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.